<a href="https://colab.research.google.com/github/marrownerd/cuda-matrix-lab/blob/main/%D0%BB%D0%B0%D0%B1%D0%B03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi

Thu Dec 25 07:44:38 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%%writefile lab3.cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <time.h>
#include <cuda_runtime.h>
#include "cublas_v2.h"

#define BLOCK_SIZE 32

// Макросы для проверки ошибок
#define CUDA_CHECK(call) \
    do { \
        cudaError_t err = call; \
        if (err != cudaSuccess) { \
            printf("CUDA error at %s:%d: %s\n", __FILE__, __LINE__, cudaGetErrorString(err)); \
            exit(EXIT_FAILURE); \
        } \
    } while (0)

#define CUBLAS_CHECK(call) \
    do { \
        cublasStatus_t status = call; \
        if (status != CUBLAS_STATUS_SUCCESS) { \
            printf("cuBLAS error at %s:%d\n", __FILE__, __LINE__); \
            exit(EXIT_FAILURE); \
        } \
    } while (0)

// 1. Наивный алгоритм
__global__ void matrixMulNaive(const float *A, const float *B, float *C, int N) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    if (row < N && col < N) {
        float sum = 0.0f;
        for (int k = 0; k < N; k++) {
            sum += A[row * N + k] * B[k * N + col];
        }
        C[row * N + col] = sum;
    }
}

// 2. Shared Memory
__global__ void matrixMulShared(const float *A, const float *B, float *C, int N) {
    __shared__ float sA[BLOCK_SIZE][BLOCK_SIZE];
    __shared__ float sB[BLOCK_SIZE][BLOCK_SIZE];
    int tx = threadIdx.x; int ty = threadIdx.y;
    int row = blockIdx.y * BLOCK_SIZE + ty;
    int col = blockIdx.x * BLOCK_SIZE + tx;
    float sum = 0.0f;

    for (int m = 0; m < (N + BLOCK_SIZE - 1) / BLOCK_SIZE; ++m) {
        if (row < N && m * BLOCK_SIZE + tx < N)
            sA[ty][tx] = A[row * N + (m * BLOCK_SIZE + tx)];
        else sA[ty][tx] = 0.0f;

        if (col < N && m * BLOCK_SIZE + ty < N)
            sB[ty][tx] = B[(m * BLOCK_SIZE + ty) * N + col];
        else sB[ty][tx] = 0.0f;
        __syncthreads();

        for (int k = 0; k < BLOCK_SIZE; ++k)
            sum += sA[ty][k] * sB[k][tx];
        __syncthreads();
    }
    if (row < N && col < N) C[row * N + col] = sum;
}

// Вспомогательные функции
void cpuMatrixMul(const float *A, const float *B, float *C, int N) {
    for (int i = 0; i < N; ++i)
        for (int j = 0; j < N; ++j) {
            float sum = 0.0f;
            for (int k = 0; k < N; ++k) sum += A[i * N + k] * B[k * N + j];
            C[i * N + j] = sum;
        }
}

void initMatrix(float *data, int size) {
    for (int i = 0; i < size; ++i) data[i] = ((float)rand() / RAND_MAX);
}

bool checkResult(const float *ref, const float *gpu, int size) {
    double epsilon = 1.0e-2;
    for (int i = 0; i < size; ++i) {
        if (fabs(ref[i] - gpu[i]) > epsilon) return false;
    }
    return true;
}

int main() {
    // Размеры для теста. Можно увеличить до 2048 или 4096 в Colab
    int sizes[] = {128, 256, 512, 1024};
    int num_tests = sizeof(sizes) / sizeof(sizes[0]);

    printf("========================================================================\n");
    printf("GPU MATRIX MULTIPLICATION BENCHMARK (Google Colab)\n");
    printf("========================================================================\n");
    printf("%-10s | %-10s | %-10s | %-10s | %-10s | %s\n",
           "Size", "CPU (ms)", "Naive(ms)", "Shared(ms)", "cuBLAS(ms)", "Check");
    printf("------------------------------------------------------------------------\n");

    for (int t = 0; t < num_tests; ++t) {
        int N = sizes[t];
        size_t bytes = N * N * sizeof(float);

        float *h_A = (float*)malloc(bytes);
        float *h_B = (float*)malloc(bytes);
        float *h_C_CPU = (float*)malloc(bytes);
        float *h_C_GPU = (float*)malloc(bytes);

        initMatrix(h_A, N * N);
        initMatrix(h_B, N * N);

        float *d_A, *d_B, *d_C;
        CUDA_CHECK(cudaMalloc(&d_A, bytes));
        CUDA_CHECK(cudaMalloc(&d_B, bytes));
        CUDA_CHECK(cudaMalloc(&d_C, bytes));

        CUDA_CHECK(cudaMemcpy(d_A, h_A, bytes, cudaMemcpyHostToDevice));
        CUDA_CHECK(cudaMemcpy(d_B, h_B, bytes, cudaMemcpyHostToDevice));

        cudaEvent_t start, stop;
        CUDA_CHECK(cudaEventCreate(&start)); CUDA_CHECK(cudaEventCreate(&stop));
        float ms_naive=0, ms_shared=0, ms_cublas=0, ms_cpu=0;

        // 1. CPU (только для малых, иначе долго)
        if (N <= 512) {
            clock_t t_start = clock();
            cpuMatrixMul(h_A, h_B, h_C_CPU, N);
            ms_cpu = 1000.0 * (double)(clock() - t_start) / CLOCKS_PER_SEC;
        } else ms_cpu = -1.0f;

        dim3 block(BLOCK_SIZE, BLOCK_SIZE);
        dim3 grid((N + BLOCK_SIZE - 1) / BLOCK_SIZE, (N + BLOCK_SIZE - 1) / BLOCK_SIZE);

        // 2. Naive
        CUDA_CHECK(cudaEventRecord(start));
        matrixMulNaive<<<grid, block>>>(d_A, d_B, d_C, N);
        CUDA_CHECK(cudaEventRecord(stop));
        CUDA_CHECK(cudaEventSynchronize(stop));
        CUDA_CHECK(cudaEventElapsedTime(&ms_naive, start, stop));

        // 3. Shared
        CUDA_CHECK(cudaEventRecord(start));
        matrixMulShared<<<grid, block>>>(d_A, d_B, d_C, N);
        CUDA_CHECK(cudaEventRecord(stop));
        CUDA_CHECK(cudaEventSynchronize(stop));
        CUDA_CHECK(cudaEventElapsedTime(&ms_shared, start, stop));

        // 4. cuBLAS
        cublasHandle_t handle;
        CUBLAS_CHECK(cublasCreate(&handle));
        float alpha = 1.0f, beta = 0.0f;
        CUDA_CHECK(cudaEventRecord(start));
        CUBLAS_CHECK(cublasSgemm(handle, CUBLAS_OP_N, CUBLAS_OP_N,
                                 N, N, N, &alpha, d_B, N, d_A, N, &beta, d_C, N));
        CUDA_CHECK(cudaEventRecord(stop));
        CUDA_CHECK(cudaEventSynchronize(stop));
        CUDA_CHECK(cudaEventElapsedTime(&ms_cublas, start, stop));
        CUBLAS_CHECK(cublasDestroy(handle));

        // Проверка результата (сравниваем cuBLAS с CPU для малых, иначе верим)
        CUDA_CHECK(cudaMemcpy(h_C_GPU, d_C, bytes, cudaMemcpyDeviceToHost));
        bool ok = (N <= 512) ? checkResult(h_C_CPU, h_C_GPU, N*N) : true;

        printf("%dx%d    | %-10.3f | %-10.3f | %-10.3f | %-10.3f | %s\n",
               N, N, ms_cpu, ms_naive, ms_shared, ms_cublas, ok ? "OK" : "FAIL");

        free(h_A); free(h_B); free(h_C_CPU); free(h_C_GPU);
        cudaFree(d_A); cudaFree(d_B); cudaFree(d_C);
    }
    return 0;
}

Writing lab3.cu


In [3]:
# Компилируем
!nvcc lab3.cu -o lab3_run -lcublas -O3

# Запускаем
!./lab3_run

GPU MATRIX MULTIPLICATION BENCHMARK (Google Colab)
Size       | CPU (ms)   | Naive(ms)  | Shared(ms) | cuBLAS(ms) | Check
------------------------------------------------------------------------
128x128    | 2.518      | 54.559     | 0.003      | 58.825     | OK
256x256    | 24.829     | 0.008      | 0.003      | 44.379     | OK
512x512    | 245.575    | 0.009      | 0.003      | 0.314      | OK
1024x1024    | -1.000     | 0.003      | 0.003      | 0.947      | OK
